# TERRA - DS Part 1 - Data Ingestion
**Sources:** Open-Meteo, Eurostat, World Bank API, GDELT Project

**Year range:** 2010–2023

**Locations:** EU 27 member countries and their capitals


In [1]:
import pandas as pd
import numpy as np
import requests
import warnings
import os
warnings.filterwarnings("ignore")

# shared constraints used by each data source
EU27_ISO2 = [
    "AT","BE","BG","CY","CZ","DE","DK","EE","ES","FI",
    "FR","GR","HR","HU","IE","IT","LT","LU","LV","MT",
    "NL","PL","PT","RO","SE","SI","SK"
]

EU27_CAPITALS = {
    "AT": ("Vienna", 48.2092, 16.3728),
    "BE": ("Brussels", 50.8503, 4.3517),
    "BG": ("Sofia", 42.6977, 23.3219),
    "CY": ("Nicosia", 35.1667, 33.3667),
    "CZ": ("Prague", 50.0755, 14.4378),
    "DE": ("Berlin", 52.5200, 13.4050),
    "DK": ("Copenhagen", 55.6761, 12.5683),
    "EE": ("Tallinn", 59.4370, 24.7536),
    "ES": ("Madrid", 40.4168, -3.7038),
    "FI": ("Helsinki", 60.1699, 24.9384),
    "FR": ("Paris", 48.8566, 2.3522),
    "GR": ("Athens", 37.9838, 23.7275),
    "HR": ("Zagreb", 45.8150, 15.9819),
    "HU": ("Budapest", 47.4979, 19.0402),
    "IE": ("Dublin", 53.3498, -6.2603),
    "IT": ("Rome", 41.9028, 12.4964),
    "LT": ("Vilnius", 54.6872, 25.2797),
    "LU": ("Luxembourg", 49.6117, 6.1319),
    "LV": ("Riga", 56.9460, 24.1059),
    "MT": ("Valletta", 35.8997, 14.5147),
    "NL": ("Amsterdam", 52.3676, 4.9041),
    "PL": ("Warsaw", 52.2297, 21.0122),
    "PT": ("Lisbon", 38.7223, -9.1393),
    "RO": ("Bucharest", 44.4268, 26.1025),
    "SE": ("Stockholm", 59.3293, 18.0686),
    "SI": ("Ljubljana", 46.0569, 14.5058),
    "SK": ("Bratislava", 48.1486, 17.1077),
}

START_YEAR = 2010
END_YEAR   = 2023
RAW_DIR    = "../data/raw"
os.makedirs(RAW_DIR, exist_ok=True) # ensure output directory exists

print(f"{len(EU27_ISO2)} countries loaded. {len(EU27_CAPITALS)} capitals loaded.")

27 countries loaded. 27 capitals loaded.


# Source 1: Open Metro

Purpose: Provide historical (2010-2022) climate records of European capital cities' climate distress indicators: 

Variables:
- Temperature mean
- Temperature anomalies
- Headwave days
- Precipitation toal
- Precipitation heavy days
- Dry days
- Evaporation trans totals 

Output:
`open_metro_raw.csv`

In [5]:
import openmeteo_requests
import requests_cache
from retry_requests import retry
import pandas as pd
import time

om = openmeteo_requests.Client(
    session=retry(requests_cache.CachedSession('.cache', expire_after=-1), retries=5, backoff_factor=0.2)
)

variables = [
    "temperature_2m_mean",
    "precipitation_sum",
    "et0_fao_evapotranspiration"
]

records = []
fetched = {agg["country_code"].iloc[0] for agg in records}

for iso2, (city, lat, lon) in EU27_CAPITALS.items():
    if iso2 in fetched:
        print(f"  Skipping {city}...")  # prevent repeat fetches
        continue
    
    print(f"Fetching {city}...", end=" ")
    try:
        r = om.weather_api(
            "https://archive-api.open-meteo.com/v1/archive",
            params={
                "latitude": lat,
                "longitude": lon,
                "daily": variables,
                "start_date": "2010-01-01",
                "end_date": "2023-12-31",
            }
        )[0].Daily()

        df = pd.DataFrame({
            "date":  pd.date_range(
                         start=pd.to_datetime(r.Time(), unit="s", utc=True),
                         periods=r.Variables(0).ValuesAsNumpy().shape[0],
                         freq="D"
                     ),
            "temp": r.Variables(0).ValuesAsNumpy(),
            "precip": r.Variables(1).ValuesAsNumpy(),
            "evap": r.Variables(2).ValuesAsNumpy(),
        })

        df["year"] = df["date"].dt.year

        agg = df.groupby("year").agg(
            temp_mean          = ("temp", "mean"),
            heatwave_days      = ("temp", lambda x: (x > 30).sum()), # days above 30 celsius
            precip_total       = ("precip", "sum"),
            precip_days_heavy  = ("precip", lambda x: (x > 20).sum()), # days above 20mm classified as "heavy rain"
            dry_days           = ("precip", lambda x: (x < 1).sum()), # days under 1mm classified as "dry day"
            evapotrans_total   = ("evap", "sum"),
        ).reset_index()

        agg["country_code"] = iso2
        records.append(agg)
        print("OK")

    except Exception as e:
        print(f"FAILED — {e}")

    time.sleep(10)  # avoid OM's rate limit

df_climate = pd.concat(records).sort_values(["country_code", "year"]).reset_index(drop=True)
print(f"\nShape: {df_climate.shape}")
df_climate.head()

Fetching Vienna... OK
Fetching Brussels... OK
Fetching Sofia... OK
Fetching Nicosia... OK
Fetching Prague... OK
Fetching Berlin... OK
Fetching Copenhagen... OK
Fetching Tallinn... OK
Fetching Madrid... OK
Fetching Helsinki... OK
Fetching Paris... OK
Fetching Athens... OK
Fetching Zagreb... OK
Fetching Budapest... OK
Fetching Dublin... OK
Fetching Rome... OK
Fetching Vilnius... OK
Fetching Luxembourg... OK
Fetching Riga... OK
Fetching Valletta... OK
Fetching Amsterdam... OK
Fetching Warsaw... OK
Fetching Lisbon... OK
Fetching Bucharest... OK
Fetching Stockholm... OK
Fetching Ljubljana... OK
Fetching Bratislava... OK

Shape: (378, 8)


,year,temp_mean,heatwave_days,precip_total,precip_days_heavy,dry_days,evapotrans_total,country_code
0,2010,9.422351,0,783.700012,7,229,759.994019,AT
1,2011,10.900239,0,498.100006,2,277,873.220825,AT
2,2012,11.087264,0,552.599976,1,254,891.626465,AT
3,2013,10.479536,2,709.000000,4,237,810.211426,AT
4,2014,11.703919,0,796.700012,4,237,811.515869,AT


In [6]:
# Save raw Open Metro to .csv
df_climate.to_csv(f"../data/raw/open_meteo_raw.csv", index=False)
print("Saved as open_meteo_raw.csv")

Saved as open_meteo_raw.csv


In [ ]:
# Sanity check
print(df_climate["country_code"].nunique(), "countries")
print(df_climate["year"].min(), "-", df_climate["year"].max())
df_climate.isnull().sum()

# Source 2: Eurostat

Purpose: Provide annual displacement and migration statistics (2010–2023) for all 27 EU member states.

Variables:
- Asylum applications (by destination country)
- Refugee population

Croatia (HR) joined the EU in 2013 and doesn't have asylum data pre-2013. Proceded with NaN in dataset. 

Output:
`eurostat_raw.csv`

In [30]:
import eurostat

# asylum applications by destination country
raw = eurostat.get_data_df("migr_asyappctza")

# melt from wide (years as columns) to long format
year_cols = [c for c in raw.columns if str(c).isdigit()]
id_cols   = [c for c in raw.columns if not str(c).isdigit()]

df = raw.melt(id_vars=id_cols, value_vars=year_cols, var_name="year", value_name="asylum_applications")
df["year"] = df["year"].astype(int)

df = df[(df["citizen"] == "TOTAL") & (df["sex"] == "T") & (df["age"] == "TOTAL") & (df["applicant"] == "TOTAL")]

# rename geo column to country_code
geo_col = [c for c in df.columns if "geo" in c.lower()][0]
df = df.rename(columns={geo_col: "country_code"})

df["country_code"] = df["country_code"].replace({"EL": "GR"}) # Greece EL -> GR country code remap

# filter to EU 27 and study years
df_eurostat = df[
    df["country_code"].isin(EU27_ISO2) & 
    df["year"].between(START_YEAR, END_YEAR)
][["country_code", "year", "asylum_applications"]]

df_eurostat = df_eurostat.sort_values(["country_code", "year"]).reset_index(drop=True)

print(f"\nShape: {df_eurostat.shape}")

df_eurostat.head()


Shape: (378, 3)


,country_code,year,asylum_applications
0,AT,2010,11060.0
1,AT,2011,14455.0
2,AT,2012,17450.0
3,AT,2013,17520.0
4,AT,2014,28065.0


In [31]:
df_eurostat.to_csv("../data/raw/eurostat_raw.csv", index=False)
print("Saved as eurostat_raw.csv")

Saved as eurostat_raw.csv


# Source 3: World Bank

Purpose: Provide annual socioeconomic indicators for all 27 EU member states (2010–2023) to give economic context to displacement trends.

Variables:
- GDP per capita (USD)
- Unemployment rate (%)
- Population total
- Urban population (%)

Output:
`worldbank_raw.csv`

In [32]:
import wbdata
import pycountry
from datetime import datetime

variables = {
    "NY.GDP.PCAP.CD": "gdp_per_capita", # in USD
    "SL.UEM.TOTL.ZS": "unemployment_rate",
    "SP.POP.TOTL": "population",
    "SP.URB.TOTL.IN.ZS": "urban_pct",
}

EU27_ISO3 = [pycountry.countries.get(alpha_2=c).alpha_3 for c in EU27_ISO2]

df_wb = wbdata.get_dataframe(
    variables,
    country=EU27_ISO3,
    date=(datetime(START_YEAR, 1, 1), datetime(END_YEAR, 1, 1)),
).reset_index()

df_wb.columns = ["country", "year"] + list(variables.values())
df_wb["year"]  = df_wb["year"].astype(int)
df_wb["country_code"] = df_wb["country"].apply(
    lambda x: pycountry.countries.search_fuzzy(x)[0].alpha_2
)

df_wb = df_wb.drop(columns=["country"])
df_wb = df_wb[df_wb["country_code"].isin(EU27_ISO2)]
df_wb = df_wb.sort_values(["country_code", "year"]).reset_index(drop=True)

print(f"\nShape: {df_wb.shape}")
df_wb.head()


Shape: (378, 6)


,year,gdp_per_capita,unemployment_rate,population,urban_pct,country_code
0,2010,46611.139342,4.883,8363404.0,67.104743,AT
1,2011,51116.895352,4.637,8391643.0,67.148426,AT
2,2012,48250.405914,4.909,8429991.0,67.208591,AT
3,2013,50305.354577,5.367,8479823.0,67.298131,AT
4,2014,51314.972262,5.674,8546356.0,67.415433,AT


In [33]:
df_wb.to_csv("../data/raw/worldbank_raw.csv", index=False)
print("Saved as worldbank_raw.csv")

Saved as worldbank_raw.csv
